# FD003 RUL Model Training (Sensors + Cycle) with CAP_125
Bu notebook, verdiğin **normalize edilmiş** FD003 CSV'leri ile çalışır ve **RUL CAP=125** uygular.

**Girdi (X):** `cycle` + tüm `os1..os3` + `s1..s21` (toplam 25 feature)

**Hedef (y):** `RUL_capped = min(RUL, 125)`

**Çıktı:** `engine_id, cycle, RUL_true_raw, RUL_true_capped, RUL_pred` + metrikler (RMSE, R², MAE)


In [8]:
!pip install catboost lightgbm --quiet

# --- 0) Kurulum / İçe Aktarımlar ---
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)

In [3]:
# --- 1) Dosya Yolları ve Parametreler ---
# İstersen DATA_DIR'ı kendi klasörüne göre değiştir.
DATA_DIR = "."

TRAIN_CSV  = os.path.join(DATA_DIR, "train_FD003_full_global_zscore_nodrop.csv")
TEST_CSV   = os.path.join(DATA_DIR, "test_FD003_full_global_zscore_nodrop.csv")
SCALER_JSON = os.path.join(DATA_DIR, "fd003_full_global_zscore_nodrop_scaler.json")

RUL_CAP = 125
RANDOM_STATE = 42

print("TRAIN_CSV :", TRAIN_CSV)
print("TEST_CSV  :", TEST_CSV)
print("SCALER_JSON:", SCALER_JSON)
print("RUL_CAP   :", RUL_CAP)


TRAIN_CSV : ./train_FD003_full_global_zscore_nodrop.csv
TEST_CSV  : ./test_FD003_full_global_zscore_nodrop.csv
SCALER_JSON: ./fd003_full_global_zscore_nodrop_scaler.json
RUL_CAP   : 125


In [4]:
# --- 2) Veriyi Yükle ---
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("train shape:", train_df.shape)
print("test  shape:", test_df.shape)

display(train_df.head())


train shape: (24720, 27)
test  shape: (16596, 27)


,engine_id,cycle,os1,os2,os3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,-0.217118,1.343108,0.0,0.0,-0.187098,-0.712023,-0.780832,-0.99998,0.781573,-0.341487,-0.704771,-0.097136,-0.353472,-0.385594,-0.227589,-0.389849,0.067685,0.469732,-0.99998,-0.889346,0.0,0.0,0.488009,-0.263506,258.0
1,1,2,0.375530,-1.037498,0.0,0.0,0.080573,-0.497646,-0.775716,-0.99998,0.781573,-0.172752,-0.452062,-0.116655,-0.353472,-0.618869,-0.193798,-0.263363,0.523935,0.729186,-0.99998,-0.321634,0.0,0.0,0.005819,0.375759,257.0
2,1,3,-0.627413,-0.697411,0.0,0.0,-0.531246,-0.841237,0.116522,-0.99998,0.781573,-0.207663,-0.262530,0.306261,-0.353472,-0.652194,-0.313602,-0.453092,0.361551,-0.100407,-0.99998,-0.889346,0.0,0.0,-0.556735,-0.175054,256.0
3,1,4,-0.900943,0.322848,0.0,0.0,0.883584,-0.362559,-1.248439,-0.99998,0.781573,0.019257,-0.452062,0.022982,-0.353472,-0.585544,-0.172294,0.052851,0.142818,-0.138416,-0.99998,-0.321634,0.0,0.0,-0.114728,-0.656179,255.0
4,1,5,0.740237,-0.017238,0.0,0.0,-1.487212,0.080880,-0.697952,-0.99998,0.781573,-0.117477,-0.199353,0.602053,-0.353472,-0.885470,-0.144647,-0.263363,0.217951,-0.153289,-0.99998,-0.321634,0.0,0.0,0.608556,0.437407,254.0


In [5]:
# --- 3) Scaler JSON Doğrulama (Bilgi amaçlı) ---
# Not: CSV'ler zaten normalize edildiği için burada tekrar ölçekleme yapmıyoruz.
# Ama JSON ile hangi kolonların ölçeklendiğini teyit etmek faydalı.

if os.path.exists(SCALER_JSON):
    with open(SCALER_JSON, "r", encoding="utf-8") as f:
        scaler = json.load(f)
    feature_cols_json = scaler.get("feature_columns", [])
    print("Scaler JSON feature_columns (len):", len(feature_cols_json))
    print(feature_cols_json)
else:
    print("Scaler JSON bulunamadı (SCALER_JSON yolu yanlış olabilir).")


Scaler JSON feature_columns (len): 24
['os1', 'os2', 'os3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


In [6]:
# --- 4) Feature Set: cycle + tüm sensör/ayarlar ---
# engine_id: kimlik
# RUL: label
# NOT: CSV zaten normalize; yalnız cycle normalize edilmemiş olabilir (sorun değil).

id_col = "engine_id"
target_col = "RUL"

# Beklenen kolonlar:
expected_feature_cols = ["cycle", "os1","os2","os3"] + [f"s{i}" for i in range(1,22)]

missing = [c for c in expected_feature_cols + [id_col, target_col] if c not in train_df.columns]
if missing:
    raise ValueError(f"Eksik kolon(lar) var: {missing}")

X_train = train_df[expected_feature_cols].copy()
X_test  = test_df[expected_feature_cols].copy()

y_train_raw = train_df[target_col].astype(float).values
y_test_raw  = test_df[target_col].astype(float).values

# --- CAP_125 uygula ---
y_train = np.minimum(y_train_raw, RUL_CAP)
y_test  = np.minimum(y_test_raw,  RUL_CAP)

print("y_train_raw min/max:", float(y_train_raw.min()), float(y_train_raw.max()))
print("y_train_cap min/max:", float(y_train.min()), float(y_train.max()))
print("y_test_raw  min/max:", float(y_test_raw.min()),  float(y_test_raw.max()))
print("y_test_cap  min/max:", float(y_test.min()),  float(y_test.max()))


y_train_raw min/max: 0.0 524.0
y_train_cap min/max: 0.0 125.0
y_test_raw  min/max: 6.0 483.0
y_test_cap  min/max: 6.0 125.0


In [10]:
# --- 5) Metrik Fonksiyonları ---
def eval_metrics(y_true, y_pred, name="model"):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))  # robust RMSE (no squared arg)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mse  = float(mean_squared_error(y_true, y_pred))
    return {"model": name, "RMSE": float(rmse), "MSE": mse, "MAE": float(mae), "R2": float(r2)}

def print_metrics(m):
    print(f"{m['model']:<20} RMSE={m['RMSE']:.4f} | MAE={m['MAE']:.4f} | R2={m['R2']:.4f}")

## Modeller
Aşağıdaki modeller eğitilir:
- LightGBM Regressor
- CatBoost Regressor
- Gradient Boosting Regressor (scikit-learn)

Ardından ensemble denemeleri:
- Basit ortalama (equal-weight)
- 70/30 ağırlıklı ortalama (paper'daki ablation mantığına benzer)
- 3'lü ortalama / 3'lü ağırlıklı
- Basit stacking (Ridge meta-learner; hızlı sürüm)

> Not: Raporlanan metrikler **CAP'li** `y_true` ile hesaplanır (yani `min(RUL, 125)`).

In [11]:
# --- 6) Base Modelleri Tanımla (makaledeki tipik ayarlara yakın, pratik/CPU-dostu) ---

# LightGBM: leaf-wise boosting; hız/başarım dengesi için makul parametreler
lgbm = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# CatBoost: güçlü tabular model; verbose azaltıldı
cat = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=200
)

# GBR: paper Table 4'teki değerlere yakın
gbr = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    random_state=RANDOM_STATE
)

models = {
    "LightGBM": lgbm,
    "CatBoost": cat,
    "GBR": gbr
}


In [12]:
# --- 7) Base Modelleri Eğit ve Test'te Değerlendir (CAP'li y) ---
results = []
preds = {}

for name, model in models.items():
    print("\nTraining:", name)
    if name == "CatBoost":
        model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
    else:
        model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    preds[name] = y_pred
    m = eval_metrics(y_test, y_pred, name)
    results.append(m)
    print_metrics(m)

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)



Training: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029499 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3380
[LightGBM] [Info] Number of data points in the train set: 24720, number of used features: 19
[LightGBM] [Info] Start training from score 93.143204
LightGBM             RMSE=13.2063 | MAE=7.8867 | R2=0.7173

Training: CatBoost
0:	learn: 38.9877685	test: 30.1931284	best: 30.1931284 (0)	total: 53.5ms	remaining: 1m 46s
200:	learn: 12.3611937	test: 12.7242902	best: 12.7184068 (195)	total: 1s	remaining: 8.96s
400:	learn: 11.4828901	test: 12.7972156	best: 12.7184068 (195)	total: 1.91s	remaining: 7.63s
600:	learn: 10.6992521	test: 12.8993756	best: 12.7184068 (195)	total: 2.86s	remaining: 6.65s
800:	learn: 10.0460249	test: 12.9807261	best: 12.7184068 (195)	total: 3.81s	remaining: 5.7s
1000:	learn: 9.4973666	test: 13.0291054	best: 12.7184068 (195)	total: 4.78s	remaining: 4.77s
1200:	

,model,RMSE,MSE,MAE,R2
1,CatBoost,12.718407,161.757872,7.778347,0.737777
0,LightGBM,13.206275,174.405690,7.886732,0.717274
2,GBR,13.378401,178.981622,8.103055,0.709856


In [13]:
# --- 8) Ensemble Denemeleri ---
ens_results = []

def add_result(name, y_pred):
    m = eval_metrics(y_test, y_pred, name)
    ens_results.append(m)
    print_metrics(m)
    return m

# Equal-weight avg (3)
avg3 = (preds["LightGBM"] + preds["CatBoost"] + preds["GBR"]) / 3.0
add_result("SimpleAvg3", avg3)

# Equal-weight avg (2): LGBM + Cat (paper'da sık geçen)
avg_lgb_cat = (preds["LightGBM"] + preds["CatBoost"]) / 2.0
add_result("Avg_LGB_Cat", avg_lgb_cat)

# 70/30 combos
w = 0.7
add_result("W70_30_LGB_Cat", w*preds["LightGBM"] + (1-w)*preds["CatBoost"])
add_result("W70_30_Cat_GBR", w*preds["CatBoost"] + (1-w)*preds["GBR"])
add_result("W70_30_LGB_GBR", w*preds["LightGBM"] + (1-w)*preds["GBR"])

# Weighted3 (örnek): 0.5 / 0.3 / 0.2
weighted3 = 0.5*preds["CatBoost"] + 0.3*preds["LightGBM"] + 0.2*preds["GBR"]
add_result("Weighted3_0.5C_0.3L_0.2G", weighted3)

ens_df = pd.DataFrame(ens_results).sort_values("RMSE")
display(ens_df)


SimpleAvg3           RMSE=12.8908 | MAE=7.7250 | R2=0.7306
Avg_LGB_Cat          RMSE=12.8046 | MAE=7.7013 | R2=0.7342
W70_30_LGB_Cat       RMSE=12.9285 | MAE=7.7430 | R2=0.7290
W70_30_Cat_GBR       RMSE=12.7589 | MAE=7.7311 | R2=0.7361
W70_30_LGB_GBR       RMSE=13.1493 | MAE=7.8358 | R2=0.7197
Weighted3_0.5C_0.3L_0.2G RMSE=12.7957 | MAE=7.6926 | R2=0.7346


,model,RMSE,MSE,MAE,R2
3,W70_30_Cat_GBR,12.758877,162.788944,7.731121,0.736106
5,Weighted3_0.5C_0.3L_0.2G,12.795745,163.731099,7.692563,0.734578
1,Avg_LGB_Cat,12.804562,163.956809,7.701304,0.734212
0,SimpleAvg3,12.890786,166.172360,7.725017,0.730621
2,W70_30_LGB_Cat,12.928510,167.146368,7.743045,0.729042
4,W70_30_LGB_GBR,13.149289,172.903794,7.835819,0.719709


In [14]:
# --- 9) Basit Stacking (hızlı sürüm) ---
# Base modellerin train set üzerindeki tahminlerini meta-feature olarak kullanır.
# (Daha doğru OOF stacking istersen GroupKFold ile ek hücre ekleyebiliriz.)

train_meta = np.column_stack([
    models["LightGBM"].predict(X_train),
    models["CatBoost"].predict(X_train),
    models["GBR"].predict(X_train),
])

test_meta = np.column_stack([
    preds["LightGBM"],
    preds["CatBoost"],
    preds["GBR"],
])

meta = Ridge(alpha=1.0, random_state=RANDOM_STATE)
meta.fit(train_meta, y_train)

stack_pred = meta.predict(test_meta)
stack_m = eval_metrics(y_test, stack_pred, "Stack_Ridge_3")
print_metrics(stack_m)


Stack_Ridge_3        RMSE=13.8481 | MAE=8.3036 | R2=0.6891


In [15]:
# --- 10) En İyi Yöntemi Seç (RMSE'e göre) ---
all_results = pd.concat([results_df, ens_df, pd.DataFrame([stack_m])], ignore_index=True)
all_results = all_results.sort_values("RMSE").reset_index(drop=True)
display(all_results)

best_name = all_results.loc[0, "model"]
print("BEST:", best_name)


,model,RMSE,MSE,MAE,R2
0,CatBoost,12.718407,161.757872,7.778347,0.737777
1,W70_30_Cat_GBR,12.758877,162.788944,7.731121,0.736106
2,Weighted3_0.5C_0.3L_0.2G,12.795745,163.731099,7.692563,0.734578
3,Avg_LGB_Cat,12.804562,163.956809,7.701304,0.734212
4,SimpleAvg3,12.890786,166.172360,7.725017,0.730621
5,W70_30_LGB_Cat,12.928510,167.146368,7.743045,0.729042
6,W70_30_LGB_GBR,13.149289,172.903794,7.835819,0.719709
7,LightGBM,13.206275,174.405690,7.886732,0.717274
8,GBR,13.378401,178.981622,8.103055,0.709856
9,Stack_Ridge_3,13.848070,191.769056,8.303578,0.689126


BEST: CatBoost


In [16]:
# --- 11) Final Tahminleri Üret (BEST'e göre) ---
def get_pred_by_name(name):
    if name in preds:
        return preds[name]
    if name == "SimpleAvg3":
        return avg3
    if name == "Avg_LGB_Cat":
        return avg_lgb_cat
    if name == "W70_30_LGB_Cat":
        return 0.7*preds["LightGBM"] + 0.3*preds["CatBoost"]
    if name == "W70_30_Cat_GBR":
        return 0.7*preds["CatBoost"] + 0.3*preds["GBR"]
    if name == "W70_30_LGB_GBR":
        return 0.7*preds["LightGBM"] + 0.3*preds["GBR"]
    if name == "Weighted3_0.5C_0.3L_0.2G":
        return weighted3
    if name == "Stack_Ridge_3":
        return stack_pred
    raise ValueError(f"Bilinmeyen model adı: {name}")

y_pred_best = get_pred_by_name(best_name)

out = pd.DataFrame({
    "engine_id": test_df["engine_id"].values,
    "cycle": test_df["cycle"].values,
    "RUL_true_raw": y_test_raw,
    "RUL_true_capped": y_test,
    "RUL_pred": y_pred_best
})

display(out.head())


,engine_id,cycle,RUL_true_raw,RUL_true_capped,RUL_pred
0,1,1,276.0,125.0,124.896621
1,1,2,275.0,125.0,124.807196
2,1,3,274.0,125.0,124.843286
3,1,4,273.0,125.0,125.460661
4,1,5,272.0,125.0,124.684459


In [17]:
# --- 12) Çıktıyı Kaydet ---
OUT_CSV = os.path.join(DATA_DIR, f"fd003_predictions_{best_name}_CAP{RUL_CAP}.csv")
out.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)


Saved: ./fd003_predictions_CatBoost_CAP125.csv


## Notlar
- Bu notebook, CSV'lerin **zaten normalize** olduğunu varsayar.
- CAP=125, hem train hem test için label'a uygulanır (`min(RUL,125)`).
- İstersen bir sonraki iterasyonda **GroupKFold (engine_id)** ile OOF stacking ekleyip, paper'a daha yakın şekilde leakage-safe stacking yaparız.

In [18]:
# --- 13) Metrikleri Dosyaya Yaz (CSV / JSON) ---
# all_results: base + ensemble + stacking metrikleri (RMSE'e göre sıralı)
# DATA_DIR: çıktıları yazacağımız klasör
from pathlib import Path
import json

metrics_df = all_results.copy()

# Bazı önceki hücrelerde MSE yoksa ekle
if "MSE" not in metrics_df.columns:
    metrics_df["MSE"] = metrics_df["RMSE"] ** 2

# Kolon düzeni (varsa)
cols = [c for c in ["model", "RMSE", "MSE", "MAE", "R2"] if c in metrics_df.columns]
metrics_df = metrics_df[cols].sort_values("RMSE").reset_index(drop=True)

display(metrics_df.head(15))

out_dir = Path(DATA_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

csv_path  = out_dir / f"metrics_fd003_CAP{RUL_CAP}.csv"
json_path = out_dir / f"metrics_fd003_CAP{RUL_CAP}.json"
best_path = out_dir / f"best_fd003_CAP{RUL_CAP}.json"

metrics_df.to_csv(csv_path, index=False)
metrics_df.to_json(json_path, orient="records", force_ascii=False, indent=2)

best_row = metrics_df.iloc[0].to_dict()
best_path.write_text(json.dumps(best_row, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved metrics CSV :", csv_path)
print("Saved metrics JSON:", json_path)
print("Saved BEST JSON   :", best_path)

# (Opsiyonel) Tahmin dosyanı da burada hatırlatalım:
print("Predictions file  :", OUT_CSV)

,model,RMSE,MSE,MAE,R2
0,CatBoost,12.718407,161.757872,7.778347,0.737777
1,W70_30_Cat_GBR,12.758877,162.788944,7.731121,0.736106
2,Weighted3_0.5C_0.3L_0.2G,12.795745,163.731099,7.692563,0.734578
3,Avg_LGB_Cat,12.804562,163.956809,7.701304,0.734212
4,SimpleAvg3,12.890786,166.172360,7.725017,0.730621
5,W70_30_LGB_Cat,12.928510,167.146368,7.743045,0.729042
6,W70_30_LGB_GBR,13.149289,172.903794,7.835819,0.719709
7,LightGBM,13.206275,174.405690,7.886732,0.717274
8,GBR,13.378401,178.981622,8.103055,0.709856
9,Stack_Ridge_3,13.848070,191.769056,8.303578,0.689126


Saved metrics CSV : metrics_fd003_CAP125.csv
Saved metrics JSON: metrics_fd003_CAP125.json
Saved BEST JSON   : best_fd003_CAP125.json
Predictions file  : ./fd003_predictions_CatBoost_CAP125.csv
